# SCF Phase 2 shard 02

Sweeps: correctness, robustness, scaling. Jobs: 13. Projected: 4.6 h
(safety-factor 1.8x applied). Grids hash: `68970a975545`.
Code source: github.com/hugogobato/scf-confounding-frontier @ tag `phase2-freeze` (pinned for
reproducibility).
Pre-registration: `docs/phase2_preregistration.md` (thresholds frozen before
any data generation; deviation register D1-D7 included there).

Resume-safe: completed cells are skipped on rerun (checkpoint parquet per
cell). If the notebook approaches the Colab wall limit it finishes the
current cell and stops cleanly; rerun to continue.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
!pip install -q "numpy>=2.0" "scipy>=1.14" "pandas>=2.2" "pyarrow>=16" scikit-learn

In [ ]:
!git clone --depth 1 --branch phase2-freeze \
    https://github.com/hugogobato/scf-confounding-frontier.git scf_repo
import sys, hashlib, json
sys.path.insert(0, "scf_repo/code")
# verify the pinned code matches the manifest recorded at generation time
EXPECTED = json.loads("{\"de_formulas.py\": \"5dffb441b638\", \"simulator.py\": \"ef31ca2a201b\", \"estimators.py\": \"7e27f25b2330\", \"detection.py\": \"06586fe60b9f\", \"runners.py\": \"df67486b60f5\"}")
for fname, short in EXPECTED.items():
    h = hashlib.sha256(open(f"scf_repo/code/{fname}", "rb").read()).hexdigest()[:12]
    assert h == short, f"code mismatch: {fname} ({h} != {short})"
print("code verified against generation-time hashes")

In [ ]:
import json, time, traceback
from multiprocessing import Pool
from runners import run_cell

JOBS = json.loads("[{\"config\": {\"n\": 2000, \"p\": 10000, \"r\": 5, \"l\": [6.708203932499369, 6.708203932499369, 6.708203932499369, 6.708203932499369, 6.708203932499369], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"5aebc7b302bf\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/5aebc7b302bf.parquet\", \"means_path\": \"data/sim/correctness/means/5aebc7b302bf.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 10000, \"r\": 25, \"l\": [6.708203932499369, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"cd5475cde3b9\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/cd5475cde3b9.parquet\", \"means_path\": \"data/sim/correctness/means/cd5475cde3b9.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 10000, \"r\": 5, \"l\": [1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 0.0, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"1c4319e3304a\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/1c4319e3304a.parquet\", \"means_path\": \"data/sim/correctness/means/1c4319e3304a.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 10000, \"r\": 5, \"l\": [6.708203932499369, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 0.0, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"a98108e0364e\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/a98108e0364e.parquet\", \"means_path\": \"data/sim/correctness/means/a98108e0364e.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"V4_corr_f\", \"corr_factors\": true, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"e513265c37d1\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/e513265c37d1.parquet\", \"means_path\": \"data/sim/robustness/means/e513265c37d1.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"V6_sparse_conf\", \"conf_kind\": \"sparse\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"966b20373966\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/966b20373966.parquet\", \"means_path\": \"data/sim/robustness/means/966b20373966.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"V6_sparse_conf\", \"conf_kind\": \"sparse\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"28f05b342fac\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/28f05b342fac.parquet\", \"means_path\": \"data/sim/robustness/means/28f05b342fac.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"V5_r-1\", \"r_misspec\": -1, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"8256b347ce61\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 125, \"raw_path\": \"data/sim/robustness/raw/8256b347ce61.parquet\", \"means_path\": \"data/sim/robustness/means/8256b347ce61.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"V5_r+1\", \"r_misspec\": 1, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"5ddae9c03815\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 125, \"raw_path\": \"data/sim/robustness/raw/5ddae9c03815.parquet\", \"means_path\": \"data/sim/robustness/means/5ddae9c03815.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 4000, \"p\": 8000, \"r\": 5, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"g\": 1.0, \"profile\": \"mixed\", \"label\": \"scaling\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"a684c471455d\", \"mode\": \"correctness\", \"sweep\": \"scaling\", \"reps\": 2, \"raw_path\": \"data/sim/scaling/raw/a684c471455d.parquet\", \"means_path\": \"data/sim/scaling/means/a684c471455d.npz\", \"_rank\": 8, \"_sweep\": \"scaling\"}, {\"config\": {\"n\": 2000, \"p\": 8000, \"r\": 5, \"l\": [6.0, 1.0, 1.0, 1.0, 1.0], \"theta\": 0.5235987755982988, \"g\": 1.0, \"profile\": \"mixed\", \"label\": \"scaling\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"36b42706d1e0\", \"mode\": \"correctness\", \"sweep\": \"scaling\", \"reps\": 2, \"raw_path\": \"data/sim/scaling/raw/36b42706d1e0.parquet\", \"means_path\": \"data/sim/scaling/means/36b42706d1e0.npz\", \"_rank\": 8, \"_sweep\": \"scaling\"}, {\"config\": {\"n\": 8000, \"p\": 1600, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"g\": 1.0, \"profile\": \"mixed\", \"label\": \"scaling\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"ef6ab884f7a8\", \"mode\": \"correctness\", \"sweep\": \"scaling\", \"reps\": 2, \"raw_path\": \"data/sim/scaling/raw/ef6ab884f7a8.parquet\", \"means_path\": \"data/sim/scaling/means/ef6ab884f7a8.npz\", \"_rank\": 8, \"_sweep\": \"scaling\"}, {\"config\": {\"n\": 1000, \"p\": 1000, \"r\": 5, \"l\": [3.0, 0.5, 0.5, 0.5, 0.5], \"theta\": 0.5235987755982988, \"g\": 1.0, \"profile\": \"mixed\", \"label\": \"scaling\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"130bb09155ff\", \"mode\": \"correctness\", \"sweep\": \"scaling\", \"reps\": 2, \"raw_path\": \"data/sim/scaling/raw/130bb09155ff.parquet\", \"means_path\": \"data/sim/scaling/means/130bb09155ff.npz\", \"_rank\": 8, \"_sweep\": \"scaling\"}]")

def _safe(job):
    try:
        return run_cell(job)
    except Exception as e:
        print('[FAIL]', job['config_id'], repr(e))
        traceback.print_exc()
        return job['config_id'], -1.0

t0 = time.time()
results = []
for i, job in enumerate(JOBS):
    if time.time() - t0 > 8.6 * 3600:
        print('[WALL LIMIT] stopping cleanly after', i, 'jobs')
        break
    results.append(_safe(job))
print('shard done:', results)

In [ ]:
import hashlib, json, glob, os
manifest = {'shard_id': 2, 'files': {}}
os.makedirs('data', exist_ok=True)
for f in sorted(glob.glob('data/**/*.parquet', recursive=True)) + \
         sorted(glob.glob('data/**/*.npz', recursive=True)):
    h = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    manifest['files'][f] = h
with open('data/manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=1)
print(json.dumps(manifest['files'], indent=1))

In [ ]:
import shutil
archive = shutil.make_archive('scf_shard_{:02d}'.format(2), 'zip', 'data')
print('archived:', archive)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)